# Spotify Classification Models

In [ ]:
cd '/Users/wiseer/Documents/github/listen-wiseer/src/'

In [ ]:
pip install lightgbm

In [ ]:
pip install imbalanced-learn

In [ ]:
import ast
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.cluster.hierarchy import dendrogram, linkage
from sklearn.metrics import classification_report, accuracy_score
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE
from utils.const import *
from analysis.data import *
from models.classifiers import *

data = LoadPlaylistData()

# # prepare spotify data
# df = pd.read_csv(
#     "/Users/wiseer/Documents/github/listen-wiseer/src/data/spotify_data.csv",
#     index_col=0,
# )
#
# # Prepare categorical variables
# df["release_date"] = pd.to_datetime(df["release_date"], format="ISO8601")
# df["year"] = df["release_date"].dt.year
# df["decade"] = df["year"].apply(lambda x: str(x)[:3] + "0s")
#
# # map keys/mode to labels
# keys = {
#     0: "C",
#     1: "Db",
#     2: "D",
#     3: "Eb",
#     4: "E",
#     5: "F",
#     6: "F#",
#     7: "G",
#     8: "Ab",
#     9: "A",
#     10: "Bb",
#     11: "B",
# }
# df["key"] = pd.to_numeric(df["key"], errors="coerce")
# df["key_labels"] = df["key"].map(keys)
# modes = {0: "Minor", 1: "Major"}
# df["mode"] = pd.to_numeric(df["mode"], errors="coerce")
# df["mode_labels"] = df["mode"].map(modes)
# df["key_mode"] = df["key_labels"] + " " + df["mode_labels"]
#
# # merge with enoa genre coordinates
# enoa_xy = pd.read_csv(
#     "/Users/wiseer/Documents/github/listen-wiseer/src/data/genres/genre_xy.csv",
#     index_col=0,
# )
# new_tracks = df.merge(enoa_xy, on="first_genre")
# # NOTE: cannot use genre mapping with this group bc of bizarre labelling
#
# new_tracks['y_target'] = 0
# new_tracks.to_csv("/Users/wiseer/Documents/github/listen-wiseer/src/data/spotify_train_data.csv")
#

In [ ]:
        df = pd.read_csv("data/spotify_train_data.csv", index_col=0)

        # filter genres I don't want recommended
        drop_genres = [
            "gospel",
            "emo",
            "k-pop",
            "comedy",
            "country",
            "sleep",
            "sertanejo",
            "hardcore",
            "industrial",
            "grindcore",
            "ska",
            "hardstyle",
            "breakbeat",
            "metal",
            "metalcore",
            "trance",
            "dub",
            "dubstep",
        ]
        df = df[~df.first_genre.isin(drop_genres)]

        # NOTE: this method loses some of the original tracks in playlist; drop duplicate first and save training data?
        df = df.drop_duplicates(["track_name", "artist_names"])


In [ ]:
    def prepare_y_target(playlist_name):
        df = data.return_enoa_data()
        # Convert artist name to single artist
        df["artist_names"] = df["artist_names"].apply(ast.literal_eval)
        df["artist_names"] = df["artist_names"].apply(
            lambda x: x[0] if isinstance(x, list) and len(x) > 0 else None
        )
        # TODO: for each playlist, filter track recommendations by genre/artist
        # eg with top genres zouk kizomba indie electronica (or first genres?)
        df.loc[df.playlist_name == playlist_name, "y_target"] = 1
        df.y_target.fillna(0, inplace=True)
        return df

    def concat_train_data(df):
        cols = [
            "id",
            "artist_names",
            "track_name",
            "release_date",
            "first_genre",
            "danceability",
            "energy",
            "key",
            "loudness",
            "mode",
            "speechiness",
            "acousticness",
            "instrumentalness",
            "liveness",
            "valence",
            "tempo",
            "popularity",
            "duration_ms",
            "time_signature",
            "year",
            "decade",
            "key_labels",
            "mode_labels",
            "key_mode",
            "top",
            "left",
            "y_target",
        ]
        df = df[cols]

        # merge dfs
        train_df = pd.read_csv(
            "/Users/wiseer/Documents/github/listen-wiseer/src/data/spotify_train_data.csv",
            index_col=0,
        )
        train_df = train_df[cols]
        df = pd.concat([df, train_df], ignore_index=True)
        return df

    def filter_genres(df):
        # map genres
        gm = pd.read_csv(
            "/Users/wiseer/Documents/github/listen-wiseer/src/data/genres/genre_map.csv",
            index_col=0,
        )
        df = df.merge(gm, on="first_genre", how="left")

        # filter genres I don't want recommended
        drop_genres = [
            "gospel",
            "emo",
            "k-pop",
            "comedy",
            "country",
            "sleep",
            "sertanejo",
            "hardcore",
            "industrial",
            "grindcore",
            "ska",
            "hardstyle",
            "breakbeat",
            "metal",
            "metalcore",
            "trance",
            "dub",
            "dubstep",
        ]
        df = df[~df.first_genre.isin(drop_genres)]

        # NOTE: this method loses some of the original tracks in playlist; drop duplicate first and save training data?
        df.drop_duplicates(["track_name", "artist_names"])

        return df

    def filter_by_playlist_genre(df):
        df = filter_genres(df) # NOTE: save this in train data
        zouk_genres = set(df[df.y_target == 1].first_genre.tolist())
        df = df[df.first_genre.isin(zouk_genres)]
        return df

    def prepare_X_y_features(zouk_train):
        # consider moving encodingto another function
        numeric_features = num_features + [
            "top",
            "left",
        ]  # let's try this out for genre
        categorical_features = ["time_signature", "key_mode", "decade"]
        # label encoding for decade; ordinal for time_signature and key_mode
        X = zouk_train[numeric_features + categorical_features]
        one_hot_encoded = pd.get_dummies(X[categorical_features])
        X = pd.concat([X, one_hot_encoded], axis=1)
        X = X.drop(categorical_features, axis=1)
        y = np.array(zouk_train.y_target).ravel()   
        return X, y

    def smote_sampling(X, y):
        # NOTE: try SMOTE or undersampling
        sm = SMOTE(random_state=42, k_neighbors=5)
        X_smote, y_smote = sm.fit_resample(X, y)
        # X_train, X_test, y_train, y_test = train_test_split(
        #     X_smote, y_smote, test_size=0.2, random_state=42
        # )
        # rf = DecisionTreeClassifier(random_state=42)
        # rf.fit(X_train, y_train)
        # y_pred = rf.predict(X_test)
        # confusion_matrix(y_test, y_pred)
        return X_smote, y_smote

    def fit_and_evaluate(y_true, y_pred):
        """Return metrics to evaluate model performance."""
        log.info("Evaluating models")
        f1 = round(f1_score(y_true, y_pred), 2)
        roc_auc = round(roc_auc_score(y_true, y_pred), 2)
        accuracy = round(accuracy_score(y_true, y_pred), 2)
        precision = round(precision_score(y_true, y_pred), 2)
        recall = round(recall_score(y_true, y_pred), 2)
        # specificity = round(specificity_score(y_true, y_pred), 2)

        return accuracy, precision, recall, f1, roc_auc  # specificicty

    def return_model_metrics(metrics):
        results = pd.DataFrame(metrics)
        results["model"] = [
            "Logistic Regression",
            "Decision Tree",
            "Random Forest",
        ]
        results = results[["model", "accuracy", "precision", "recall", "f1", "roc_auc"]]
        return results

    def return_feature_selection(rfe_features):
        log.info("Selecting features")
        feature_selection = pd.DataFrame(rfe_features).T
        feature_selection.columns = [
            "Logistic",
            "Decision Tree",
            "Random Forest",
        ]

        return feature_selection

    def config_model_pipeline():
        """Builds pipeline for sklearn models and returns model metrics."""
        pipelines = []
        pipelines.append(
            Pipeline(
                [
                    (
                        "scaler",
                        MinMaxScaler(),
                    ),
                    (
                        "rfe",
                        RFE(
                            estimator=LogisticRegression(),
                            n_features_to_select=15,
                        ),
                    ),
                    ("classifier", LogisticRegression()),
                ]
            )
        )
        pipelines.append(
            Pipeline(
                [
                    (
                        "scaler",
                        MinMaxScaler(),
                    ),
                    (
                        "rfe",
                        RFE(
                            estimator=DecisionTreeClassifier(),
                            n_features_to_select=15,
                        ),
                    ),
                    ("classifier", DecisionTreeClassifier()),
                ]
            )
        )
        pipelines.append(
            Pipeline(
                [
                    (
                        "scaler",
                        MinMaxScaler(),
                    ),
                    (
                        "rfe",
                        RFE(
                            estimator=RandomForestClassifier(n_estimators=100),
                            n_features_to_select=15,
                        ),
                    ),
                    ("classifier", RandomForestClassifier(n_estimators=100)),
                ]
            )
        )
        return pipelines

    def compare_classification_models(X,y):
        # logistic, decision tree and random forest
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.3, random_state=0
        )
        pipelines = config_model_pipeline()
        metrics = []
        for model in pipelines:
            model.fit(X_train, y_train)
            y_true = y_test
            y_pred = model.predict(X_test)
            accuracy, precision, recall, f1, roc_auc = fit_and_evaluate(
                y_true, y_pred
            )
            metrics.append(
                {
                    "accuracy": accuracy,
                    "precision": precision,
                    "recall": recall,
                    "f1": f1,
                    "roc_auc": roc_auc,
                }
            )
        metrics = return_model_metrics(metrics)
        return metrics


In [ ]:
# prepare data
df = prepare_y_target()  # based on playlist
df = concat_train_data(df)

In [ ]:
# for each playlist, loop
df = filter_by_playlist_genre(df)
print("sample size: ", round(100*len(df[df.y_target==1])/len(df),1), "%")

# NOTE: can either improve sampling with SMOTE or reduce genre filtering method for recommendations

## Method 1 

In [ ]:
# Method 1: filter by playlist and run individual model
X, y = prepare_X_y_features(df)
metrics = compare_classification_models(X, y)

# NOTE: tree-based and boosting algorithms are ideal for imbalanced data bc weight is given to minority class at each successive iteration
metrics

In [ ]:
## Optimize decision tree classifier
# from sklearn.model_selection import GridSearchCV
# from sklearn.tree import DecisionTreeClassifier
# 
# X_train, X_test, y_train, y_test = train_test_split(
#     X, y, test_size=0.3, random_state=0
# )
# 
# # Perform grid search for decision tree
# param_grid = [
#     {
#         "max_depth": [1, 2, 3, 4, 5, 6],
#         "criterion": ["entropy", "gini"],
#         "splitter": ["best", "random"],
#     }
# ]
# tree = GridSearchCV(DecisionTreeClassifier(), param_grid)
# tree.fit(X_train, y_train)
# 
# # Print grid search results
# print("Grid search mean and stdev:\n")
# scoring = tree.cv_results_
# 
# # Print best params
# print(classification_report(y_test, tree.predict(X_test)))
# print("\nBest parameters:", tree.best_params_)
# best_max_depth = tree.best_params_["max_depth"]

In [ ]:
df

In [ ]:
X_test

In [ ]:
results = df[["id", "artist_names", "track_name", "first_genre"]].merge(
    X_test, left_index=True, right_index=True, how="inner"
)
# only identifies fraction of songs and they are all from the original songs
# TODO: filter original songs from pred list to add
#results[results.y_pred == 1]

In [ ]:
results[results.y_pred>0]

In [ ]:
results[results.y_pred < 0]

In [ ]:

import lightgbm as lgb

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Convert the dataset into LightGBM format
train_data = lgb.Dataset(X_train, label=y_train)
test_data = lgb.Dataset(X_test, label=y_test, reference=train_data)

# Set LightGBM parameters
params = {
    # "objective": "multiclass",
    # "num_class": 18,  # Number of classes
    # "metric": "multi_logloss",  # Logarithmic Loss for multiclass
    "boosting_type": "gbdt",  # Gradient Boosting Decision Tree
    "num_leaves": 31,
    "learning_rate": 0.05,
    "feature_fraction": 0.9,
    "force_row_wise": True,
}

# Train the model
num_round = 100
bst = lgb.train(params, train_data, num_round, valid_sets=[test_data])

# Make predictions
y_pred = bst.predict(X_test, num_iteration=bst.best_iteration)
y_pred_class = [np.argmax(pred) for pred in y_pred]

# Evaluate the model
accuracy = accuracy_score(y_test, y_pred_class)
classification_rep = classification_report(y_test, y_pred_class)

# Print results
print(f"Accuracy: {accuracy}")
print("Classification Report:\n", classification_rep)

In [ ]:
X_test.y_pred.value_counts()

In [ ]:
X_test["y_pred"] = y_pred_class

In [ ]:
results = df[["id", "artist_names", "track_name", "first_genre"]].merge(
    X_test, left_index=True, right_index=True, how="inner"
)
# only identifies fraction of songs and they are all from the original songs
# TODO: filter original songs from pred list to add
results[results.y_pred == 1]

In [ ]:
df

In [ ]:
model = IsolationForest(
    n_estimators=150, max_samples="auto", contamination=0.05, max_features=10
).fit(X_train).predict(X_train)

In [ ]:
model

In [ ]:
# Train the isolation forest model
if_model = IsolationForest(n_estimators=100, random_state=0).fit(X_train)
# Predict the anomalies
if_prediction = if_model.predict(X_test)
# Change the anomalies' values to make it consistent with the true values
if_prediction = [1 if i == -1 else 0 for i in if_prediction]
# Check the model performance
print(classification_report(y_test, if_prediction))

In [ ]:
model = IsolationForest(n_estimators=150, max_samples="auto", contamination=0.05, max_features=10).fit(X_train)
df['iso_scores'] = model.decision_function(X)
df.iso_scores.hist()

In [ ]:
df[df.iso_scores<0]

In [ ]:
df[df.iso_scores > 0]

In [ ]:
# Use updated with best params
tree_optimized = DecisionTreeClassifier(
    criterion="gini", max_depth=best_max_depth, splitter="best"
)
_opt = tree_optimized.fit(X_train, y_train)

print(classification_report(y_test, tree_optimized.predict(X_test)))

    # TODO: k-fold validation; features importance; optimization
    #X_smote, y_smote = smote_sampling(X, y)

    def return_feature_selection(rfe_features):
        log.info("Selecting features")
        feature_selection = pd.DataFrame(rfe_features).T
        feature_selection.columns = [
            "Logistic",
            "Decision Tree",
            "Random Forest",
        ]

        return feature_selection

In [ ]:
X_test["y_pred"] = y_pred
results = zouk_train[
    ["id", "artist_names", "track_name", "first_genre"]
].merge(X_test, left_index=True, right_index=True, how="inner")
# only identifies fraction of songs and they are all from the original songs
# TODO: filter original songs from pred list to add
results[results.y_pred==1]

In [ ]:
results[results.y_pred == 0][:60]

In [ ]:
results[results.y_pred == 0]

In [ ]:
pd.crosstab(results.first_genre, results.y_pred)

In [ ]:
#        # add feature importance
#        X_train, X_test, y_train, y_test = train_test_split(
#            X, y, test_size=0.3, random_state=0
#        )
#
#        pipelines = config_model_pipeline()
#        metrics = []
#        rfe_features = []
#        rfe_values = []
#        for model in pipelines:
#            model.fit(X_train, y_train)
#            rfe_features = model.named_steps["rfe"].support_
#            rfe_features = list(X.columns[rfe_features])
#            rfe_features.append(rfe_features)
#            rfe_values = list(model.named_steps["rfe"].estimator_.feature_importances_)
#            rfe_values.append(rfe_values)
#            y_true = y_test
#            y_pred = model.predict(X_test)
#            accuracy, precision, recall, f1, roc_auc = fit_and_evaluate(
#                y_true, y_pred
#            )
#            metrics.append(
#                {
#                    "accuracy": accuracy,
#                    "precision": precision,
#                    "recall": recall,
#                    "f1": f1,
#                    "roc_auc": roc_auc,
#                }
#            )
#        metrics = return_model_metrics(metrics)
#        feature_selection = return_feature_selection(rfe_features)

## Multinomial Models

In [ ]:
## method 2: match new music to playlist in group
## create dance labels for multinomial model
dance = {
    "kizomba": 0,
    "bachata": 1,
    "zouk": 2,
}
df["dance"] = np.where(df.my_genre.isin(["zouk", "kizomba", "indie", "electronica", "ambient"]), "zouk", np.nan)
df["dance"] = np.where(df.my_genre.isin(["bachata"]), "bachata", df.dance)
df["dance"] = np.where(df.my_genre.isin(["kizomba"]), "kizomba", df.dance)
df["y_target"] = df.my_genre.map(dance)

In [ ]:
import lightgbm as lgb

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Convert the dataset into LightGBM format
train_data = lgb.Dataset(X_train, label=y_train)
test_data = lgb.Dataset(X_test, label=y_test, reference=train_data)

# Set LightGBM parameters
params = {
    "objective": "multiclass",
    "num_class": 3,  # Number of classes
    "metric": "multi_logloss",  # Logarithmic Loss for multiclass
    "boosting_type": "gbdt",  # Gradient Boosting Decision Tree
    "num_leaves": 31,
    "learning_rate": 0.05,
    "feature_fraction": 0.9,
    "force_row_wise": True,
}

# Train the model
num_round = 100
bst = lgb.train(params, train_data, num_round, valid_sets=[test_data])

# Make predictions
y_pred = bst.predict(X_test, num_iteration=bst.best_iteration)
y_pred_class = [np.argmax(pred) for pred in y_pred]

# Evaluate the model
accuracy = accuracy_score(y_test, y_pred_class)
classification_rep = classification_report(y_test, y_pred_class)

# Print results
print(f"Accuracy: {accuracy}")
print("Classification Report:\n", classification_rep)

In [ ]:
y_test

In [ ]:
X_test["y_pred"] = y_pred_class
results = zouk_train[["id", "artist_names", "track_name", "first_genre"]].merge(
    X_test, left_index=True, right_index=True, how="inner"
)
# only identifies fraction of songs and they are all from the original songs
# TODO: filter original songs from pred list to add
results[results.y_pred == 1]

In [ ]:
results[results.y_pred == 0]

In [ ]:
pd.crosstab(results.first_genre, results.y_pred)